In [ ]:
# 하이퍼 파라미터 튜닝 기법
# Parameter = 매개변수
# function가 값을 반환 | 실행문 실행 => 매개변수 값을 받아서 => 출력
# KNN, SVM, Logistic Regression -> 반지름 반경, 값 마진, 변수
# 각 모델들이 최적의 결과를 도출하기 위해서 모델들만의 최고의 매개변수값 적용!
# 이 매개변수를 어떻게, 어떤 방식으로, 어떤값을 산출?!!

# 사이킷 런 라이브러리 모듈
# 어떤 값을 파라미터 값으로 모델에 전달해야 최고의 결과값을 예측할 수 있는지
# 알려주는 기능 존재 // 하이퍼 파라미터 튜닝 기법

# GridSearch
# 최적의 하이퍼 파라미터 값 후보 리스트를 제공, 각 후보군들을 종합해서 최고의 확률(성능)을 도출할 수 있는 값의 조합을 만들어서 최종적인 확률반환

# RandomSearch
# 최적의 하이퍼 파라미터 값 범위 제공, 해담 범위 안에서 랜덤하게 값 추출 > 가장 높은 성능을 내는 최종적인 확률값 도출

# Bayesian Optimization
# 최적의 하이퍼 파라미터 값 범위 제공 > 랜덤서치 기법과 다르게, 앞서 검증했던 파라미터 값의 결과값을 반영해서 최적의 하이퍼 파라미터 값을 도출



In [2]:
import pickle
import pandas as pd

with open("titanic_step4_importance_train.pickle", "rb") as pickle_filename :
    train_importance = pickle.load(pickle_filename)
with open("titanic_step4_importance_test.pickle", "rb") as pickle_filename :
    test_importance = pickle.load(pickle_filename)
with open("titanic_step4_importance_train_y.pickle", "rb") as pickle_filename :
    train_answer = pickle.load(pickle_filename)

In [3]:
train_importance.head()

,Initial_0,Pclass_2,Sex_0,Sex_1,HighChance_0,Cabin_0,Fare_0,LowChance_0,Ticket_Num_Cut_2,Family_2,...,Ticket_Num_Cut_3,Family_4,Family_0,Age_6,Fare_3,Cabin_4,Ticket_Initial2_2,Cabin_1,Age_1,Ticket_Initial2_15
0,1,1,1,0,1,1,1,1,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,1,0,0,...,1,0,0,0,1,0,0,1,0,0
2,0,1,0,1,1,1,1,1,0,0,...,0,0,1,0,0,0,1,0,0,0
3,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
4,1,1,1,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [4]:
test_importance.head()

,Initial_0,Pclass_2,Sex_0,Sex_1,HighChance_0,Cabin_0,Fare_0,LowChance_0,Ticket_Num_Cut_2,Family_2,...,Ticket_Num_Cut_3,Family_4,Family_0,Age_6,Fare_3,Cabin_4,Ticket_Initial2_2,Cabin_1,Age_1,Ticket_Initial2_15
0,1,1,1,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,1,0,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,1,1,0,0,0,...,0,0,1,1,0,0,0,0,0,0
3,1,1,1,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,1,0,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
train_answer.head()

,Survived
0,0.0
1,1.0
2,1.0
3,1.0
4,0.0


In [36]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 1.2 MB/s eta 0:00:59
   ---------------------------------------- 0.8/69.5 MB 1.2 MB/s eta 0:00:59
    --------------------------------------- 1.0/69.5 MB 1.2 MB/s eta 0:00:59
    --------------------------------------- 1.3/69.5 MB 1.2 MB/s eta 0:00:58
    --------------------------------------- 1.6/69.5 MB 1.2 MB/s eta 0:00:58
   - -------------------------------------- 1.8/69.5 MB 1.2 MB/s eta 0:00:58
   - -------------------------------------- 2.1/69.5 MB 1.2 MB/s eta 0:00:58
   - -------------------------------------- 2.4/69.5 MB 1.2 MB/s eta 0:00:57
   - -------------------------

In [39]:
pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.2 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [40]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy import stats

from sklearn.neighbors import KNeighborsClassifier        # 1. KNN 
from sklearn.linear_model import LogisticRegression       # 2. Logistic Regression
from sklearn.svm import SVC                               # 3. SVC
from sklearn.tree import DecisionTreeClassifier           # 4. Decision Tree
from sklearn.ensemble import RandomForestClassifier       # 5. Random Forest
from sklearn.ensemble import ExtraTreesClassifier         # 6. Extra Tree
from sklearn.ensemble import GradientBoostingClassifier   # 7. GBM
from sklearn.naive_bayes import GaussianNB                # 8. GausianNB
from xgboost import XGBClassifier                         # 9. XGBoost
from lightgbm import LGBMClassifier                       # 10. LightGBM

In [ ]:
# 지금까지 설명했던 3가지 하이퍼 파라미터 튜닝기법들 중 1개 혹은 복수의 기법을 선택
# 정확도 (확률) 도출
# 모델 : SVM > SVC 모델
# SVC 모델 > 2가지의 하이퍼파라미터 값이 필요
# C : regularization 파라미터
# gamma : 어느 정도의 훈련세트에서 fit(학습)하게 할지를 결정

In [ ]:
# C (Regularization Parameter)
# 오분류(Misclassfication)를 얼마나 허용할 것인가를 결정하는 값
# 실수를 얼마나 용서할 것인가?
# gamma : 주변에 얼만큼 민감하게 영향을 받게 할 것인가?

In [8]:
hyerparams = {
    "C": stats.uniform(0, 50),
    "gamma": stats.uniform(0, 1)
}

gd = RandomizedSearchCV(
    estimator=SVC(random_state=1),
    param_distributions=hyerparams,
    n_iter=100,  #100번 검증
    cv=5, 
    scoring="accuracy",
    random_state=1,
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

RandomizedSearchCV(cv=5, estimator=SVC(random_state=1), n_iter=100, n_jobs=-1,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D3C8EDA7B0>,
                                        'gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D3C9CDB390>},
                   random_state=1, scoring='accuracy')

In [11]:
print(gd.best_score_)
print(gd.best_params_)

0.8395204318624068
{'C': np.float64(2.995884475610583), 'gamma': np.float64(0.12134345574073735)}


In [12]:
df = pd.DataFrame(gd.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.033463,0.000644,0.016339,0.001055,20.851100,0.720324,"{'C': 20.8511002351287, 'gamma': 0.72032449344...",0.765363,0.803371,0.842697,0.820225,0.825843,0.811500,0.026267,71
1,0.034711,0.005191,0.017777,0.000957,0.005719,0.302333,"{'C': 0.005718740867244332, 'gamma': 0.3023325...",0.614525,0.617978,0.617978,0.617978,0.612360,0.616163,0.002325,100
2,0.029433,0.003256,0.019154,0.011577,7.337795,0.092339,"{'C': 7.337794540855652, 'gamma': 0.0923385947...",0.798883,0.814607,0.853933,0.831461,0.853933,0.830563,0.021686,9
3,0.032167,0.002528,0.014355,0.000961,9.313011,0.345561,"{'C': 9.313010568883545, 'gamma': 0.3455607270...",0.798883,0.803371,0.837079,0.820225,0.837079,0.819327,0.016147,37
4,0.031381,0.001461,0.020399,0.009332,19.838374,0.538817,"{'C': 19.838373711533496, 'gamma': 0.538816734...",0.793296,0.803371,0.842697,0.825843,0.848315,0.822704,0.021467,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.034633,0.009059,0.012174,0.000366,13.164839,0.065961,"{'C': 13.164838524355549, 'gamma': 0.065961090...",0.798883,0.825843,0.853933,0.831461,0.853933,0.832810,0.020463,7
96,0.038394,0.010484,0.018346,0.001724,36.753298,0.772178,"{'C': 36.753298164433474, 'gamma': 0.772178029...",0.765363,0.803371,0.842697,0.820225,0.825843,0.811500,0.026267,71
97,0.032380,0.001222,0.018418,0.000638,45.390793,0.931972,"{'C': 45.3907926251762, 'gamma': 0.93197206919...",0.759777,0.803371,0.837079,0.820225,0.820225,0.808135,0.026425,89
98,0.026850,0.001366,0.013973,0.000847,0.697579,0.234362,"{'C': 0.6975786487798508, 'gamma': 0.234362086...",0.832402,0.831461,0.825843,0.803371,0.870787,0.832773,0.021723,8


In [13]:
df[["params", "mean_test_score"]]

,params,mean_test_score
0,"{'C': 20.8511002351287, 'gamma': 0.72032449344...",0.811500
1,"{'C': 0.005718740867244332, 'gamma': 0.3023325...",0.616163
2,"{'C': 7.337794540855652, 'gamma': 0.0923385947...",0.830563
3,"{'C': 9.313010568883545, 'gamma': 0.3455607270...",0.819327
4,"{'C': 19.838373711533496, 'gamma': 0.538816734...",0.822704
...,...,...
95,"{'C': 13.164838524355549, 'gamma': 0.065961090...",0.832810
96,"{'C': 36.753298164433474, 'gamma': 0.772178029...",0.811500
97,"{'C': 45.3907926251762, 'gamma': 0.93197206919...",0.808135
98,"{'C': 0.6975786487798508, 'gamma': 0.234362086...",0.832773


In [14]:
#Grid Search : [] 안에 직접 값을 기입해서 입력한 값의 경우의 수를 모두 따지는 것

hyperparams = {
    "C": [10, 15, 20, 23, 25, 30, 50],
    "gamma": [0.001, 0.01, 0.05, 0.06, 0.07, 1]
}

gd = GridSearchCV(
    estimator=SVC(random_state=1),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

Fitting 5 folds for each of 42 candidates, totalling 210 fits


GridSearchCV(cv=5, estimator=SVC(random_state=1), n_jobs=-1,
             param_grid={'C': [10, 15, 20, 23, 25, 30, 50],
                         'gamma': [0.001, 0.01, 0.05, 0.06, 0.07, 1]},
             scoring='accuracy', verbose=True)

In [15]:
print(gd.best_score_)
print(gd.best_params_)

0.8361747536250078
{'C': 15, 'gamma': 0.07}


In [16]:
train_answer.head()

,Survived
0,0.0
1,1.0
2,1.0
3,1.0
4,0.0


In [17]:
learning_rate = [0.01, 0.05, 0.1, 0.2] # 학습속도 (1에 수렴할수록 최대한 빨리 한번에 많이 학습)
n_estimators = [100, 1000, 2000]  #몇번을 반복할 것인가
max_depth = [3, 5, 10, 15] #의사결정 트리의 가짓수, 밑으로가 몇번을 할거냐 4가지 옵션 제공

hyperparams = {
    "learning_rate": learning_rate, 
    "n_estimators": n_estimators,
    "max_depth": max_depth
}

gd = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=1), #
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

Fitting 5 folds for each of 48 candidates, totalling 240 fits


GridSearchCV(cv=5, estimator=GradientBoostingClassifier(random_state=1),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1, 0.2],
                         'max_depth': [3, 5, 10, 15],
                         'n_estimators': [100, 1000, 2000]},
             scoring='accuracy', verbose=True)

In [18]:
gd.best_score_

np.float64(0.8428912183792605)

In [19]:
gd.best_params_

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 1000}

In [20]:
from sklearn.model_selection import RandomizedSearchCV

hyperparams = {
    "penalty": ["l1", "l2", "elasticnet"],
    "C": stats.uniform(0, 1000)
}

gd = RandomizedSearchCV(
    estimator = LogisticRegression(random_state = 1), 
    param_distributions=hyperparams,
    n_iter=100,
    cv=5,
    scoring="accuracy",
    random_state=1, 
    n_jobs=-1
)

#학습하기
gd.fit(train_importance, train_answer.values.ravel()) 

# svc => c, gamma
# gradient classification => learning_rate, max_depth, n_estimators
# logistic regression(논리회귀) => penalty, c

# penalty => 어떤 방식으로 overfitting (과적합)을 막을 것인가?
#       -> 과적합: EX)학교 내신시험문제 공부 -> 집어준 문제는 잘푸는데 수능은 못보듯이, 이론에 대한 완전한 이해와 적용은 못함
#       -> 과적합 방지: 
# l1 = Lasso => 머신러닝 모델이 학습을 하는데, 굳이 불필요하다고 판단되는 피처 즉 변수는 처음부터 배제하고 시작
# l2 = Ridge => 머신러닝 모델이 학습을 하는데, 굳이 불필요하다고 판단되는 피처를 아예 제거하지는 않고, 가중치를 낮게 줄여
# elasticnet => l1 + l2 의 보완지점 => 누가봐도 영양가 없는 피쳐 확실히 버려 + 애매한 피쳐는 가져는가 (가중치)

# C => penalty로 선택한 과적합 방지 방식을 사용하되, 얼마나 강하게 막을 것인가? 
# C라는 값이 작을수록 강하게 penalty를 부여 > 정규화를 강하게 한다
# C = 1/lambda => 정규화 표준값,

RandomizedSearchCV(cv=5, estimator=LogisticRegression(random_state=1),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D3C9CDBC50>,
                                        'penalty': ['l1', 'l2', 'elasticnet']},
                   random_state=1, scoring='accuracy')

In [1]:
#현재 진행상황

#1. Kaffle 사이트 > raw data download (titanic)
#2. EDA > 탐색적 데이터 분석 과정 (생존, 사망 >  어떤 column 상관관계)
#3. Feature ENgineering (컴퓨터는 수치화된 데이터를 좋아하기 때문 )
#4. Model Process(KNN, Linear Regression, Logistic Regression, Baging, Boosting, Decision Tree, SVM,SVC, Random Forest 등등) 각 모델에 대한 설명
#4. Cross Validation (교차검증)
#5. High Chance & Important Features :교차검증 단계에서 모델이 학습한 결과값을 기반으로 테스트 데이터를 맞추기 전에 학습한 데이터를 가지고 다시 점검을 해봐서 score 도출
#   -> 이 score를 기반으로 도출 시, 어떤 feature에 높은 점수를 부여했는가가 나옴 > Decision Tree
#6. Hyper Parameter Tuning (각 모델들에게 전달할 매개변수 값, 검증된 방법 중 하나 , Grid Search, Random Search,)

In [22]:
gd.best_score_
#최고의 경우의 수로 실행했을때 정확도가 83%

np.float64(0.8384282217061075)

In [23]:
gd.best_params_

{'C': np.float64(846.3109166860171), 'penalty': 'l2'}

In [24]:
import numpy as np
np.linspace(700,900,10) #특점 범위구간 내에 원하는 만큼의 갯수의 난수를 생성해줌
    #-> 직접 난수를 지정해서 넣어주기 귀찮을때 
    #-> np(시작값, 종료지점, 만들고 싶은 숫자 개수)

array([700.        , 722.22222222, 744.44444444, 766.66666667,
       788.88888889, 811.11111111, 833.33333333, 855.55555556,
       877.77777778, 900.        ])

In [26]:
#GridSearch 기법에 사용할 변수 값을 만들기
penalty = ["l1","l2"]
C = np.linspace(700,900,200)

hyperparams = {
    "penalty": penalty,
    "C": C
}

#기법 적용하기
gd = GridSearchCV(
    estimator=LogisticRegression(random_state=1),
    param_grid=hyperparams,
    verbose=True, #중간 진행상황과 과정을 나에게 알려줘
    cv=5, 
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

Fitting 5 folds for each of 400 candidates, totalling 2000 fits


GridSearchCV(cv=5, estimator=LogisticRegression(random_state=1), n_jobs=-1,
             param_grid={'C': array([700.        , 701.00502513, 702.01005025, 703.01507538,
       704.0201005 , 705.02512563, 706.03015075, 707.03517588,
       708.04020101, 709.04522613, 710.05025126, 711.05527638,
       712.06030151, 713.06532663, 714.07035176, 715.07537688,
       716.08040201, 717.08542714, 718.09045226, 719.09547739,
       720...
       876.88442211, 877.88944724, 878.89447236, 879.89949749,
       880.90452261, 881.90954774, 882.91457286, 883.91959799,
       884.92462312, 885.92964824, 886.93467337, 887.93969849,
       888.94472362, 889.94974874, 890.95477387, 891.95979899,
       892.96482412, 893.96984925, 894.97487437, 895.9798995 ,
       896.98492462, 897.98994975, 898.99497487, 900.        ]),
                         'penalty': ['l1', 'l2']},
             scoring='accuracy', verbose=True)

In [27]:
gd.best_score_

np.float64(0.8384282217061075)

In [28]:
gd.best_params_

{'C': np.float64(705.0251256281407), 'penalty': 'l2'}

In [ ]:
# 하이퍼 파라미터(Grid, Random) 튜닝기법 -> SVC,lOGISTIC REGRESSION, GRDIENT BOOSTING MACHINE

In [ ]:
#XGBoost(Bayesian Optimization): 대표적으로 Boosting 기법을 사용하는 모델 중 하나 > Decision Tree

In [ ]:
# 각 모델들은 저마다의 하이퍼 파라미터를 요구한다. 

#XGBoost가 필요로 하는 하이퍼 파라미터: 

#min_child_weight : 자식 노드의 최소 가중치  
    #-> 최소 가중치보다 낮은 가중치의 질문은 하지말라는 것. 너무 한쪽으로 편향되거나 안좋은 질문으로 가지 못하게 하기 위함.
    #-> 얼마의 가중치를 줘야 최고의 score를 모델이 낼 수 있을까를 ML를 통해 알아보는 것
    #-> ex) 10 = 자식요소가 최소 10개는 나오게끔

#subsample : 현재 가지고 있는 데이터의 행을 랜덤하게 선택하는 비율
    #학습을 위해 랜덤으로 값을 섞었을때에 얼마나 데이터를 섞어서 가져올건지
    # -> ex) 0.8 => 100개중에 80개를 랜덤값으로 20개를 가지고 있는 데이터를 섞겠다

#colsample_bytree : 현재 가지고 있는 데이터의 열을 랜덤하게 선택하는 비율
    #ex) 0.5 = 전체 50개의 열 중 25개의 열만 선택해서 가져오겠다. but 반복할때마다 그 25개는 매번 50개중 랜덤하게 가져옴

#reg_alpha : L1정규화(regularization)의 가중치 
#reg_lambda : L2정규화 가중치

In [29]:
pip install bayesian-optimization

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [41]:
import numpy as np
from xgboost import XGBClassifier
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

#매개변수 정의
pbounds = {
    "learning_rate": (0.01, 0.5),
    "n_estimators": (100,1000),
    "max_depth": (3,10),
    "min_child_weight": (0,10),
    "subsample": (0.5,1.0), 
    "colsample_bytree": (0.5,1.0),
    "gamma" : (0, 5),   #-> gamma값이 높을수록 주변의 영향을 많이 받음.
    #reg_lambda,reg_alpha까지 하면 시간이 너무 오래걸려서 지금은 생략
    # "reg_lambda":(0,1000),
    # "reg_alpha": (0,1.0)
}

#매개변수를 적용할 함수 정의
def xgboost_hyper_param(learning_rate, n_estimators, max_depth, min_child_weight, subsample, colsample_bytree, gamma) : 
    max_depth=int(max_depth)
    n_estimators=int(n_estimators)
    clf=XGBClassifier(
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        random_state=1,
        eval_metic="logloss"
    )
    return np.mean(cross_val_score(clf, train_importance, train_answer, cv=5,scoring="accuracy"))

#함수 실행(선언)

optimizer = BayesianOptimization(f=xgboost_hyper_param, pbounds=pbounds, random_state=1)

optimizer.maximize(init_points=10, n_iter=100)  
#일단 10번은 너기 변수값을 랜덤으로 찾아서 score값을 내봐.
#BayesianOptimization은 여기서 끝내는게 아니라 그 10번을 가지고 나온 결과값을 기준으로, 최고의 확률을 낼 수 있는 하이퍼파라미터 값을 찾는 실행을 100번을 하는 것.

|   iter    |  target   | learni... | n_esti... | max_depth | min_ch... | subsample | colsam... |   gamma   |
-------------------------------------------------------------------------------------------------------------
| 1         | 0.8238026 | 0.2143407 | 748.29204 | 3.0008006 | 3.0233257 | 0.5733779 | 0.5461692 | 0.9313010 |
| 2         | 0.8137153 | 0.1793247 | 457.09072 | 6.7717171 | 4.1919451 | 0.8426097 | 0.6022261 | 4.3905871 |
| 3         | 0.8125918 | 0.0234199 | 703.42075 | 5.9211336 | 5.5868982 | 0.5701934 | 0.5990507 | 4.0037228 |
| 4         | 0.8260435 | 0.4844481 | 382.08176 | 7.8462583 | 8.7638915 | 0.9473033 | 0.5425221 | 0.1952739 |
| 5         | 0.8114807 | 0.0932169 | 890.32825 | 3.6884278 | 4.2110762 | 0.9789447 | 0.7665826 | 3.4593855 |
| 6         | 0.8238089 | 0.1646026 | 717.85083 | 8.8423797 | 0.1828827 | 0.8750721 | 0.9944305 | 3.7408282 |
| 7         | 0.8328165 | 0.1474175 | 810.35139 | 3.7225820 | 4.4789352 | 0.9542977 | 0.6468070 | 1.4388766 |
| 8       

In [43]:
optimizer.max

{'target': np.float64(0.8440085368150146),
 'params': {'learning_rate': np.float64(0.01),
  'n_estimators': np.float64(818.6080628195011),
  'max_depth': np.float64(6.515857404122109),
  'min_child_weight': np.float64(0.0),
  'subsample': np.float64(0.5),
  'colsample_bytree': np.float64(1.0),
  'gamma': np.float64(2.702194266388444)}}

In [53]:
#Gridsearch 적용해보기
#1
learning_rate= [0.001,0.005, 0.01, 0.05, 0.06, 0.1, 0.12, 0.15, 0.17, 0.2]
n_estimators = [10, 50, 60, 75, 85, 100, 125, 150, 200, 250, 500, 1000]

hyperparams = {
    "learning_rate": learning_rate,
    "n_estimators" : n_estimators
}

gd= GridSearchCV(
    estimator=XGBClassifier(random_state=1, eval_metric="logloss"),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuarcy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer)

InvalidParameterError: The 'scoring' parameter of GridSearchCV must be a str among {'matthews_corrcoef', 'adjusted_mutual_info_score', 'd2_absolute_error_score', 'roc_auc_ovr_weighted', 'f1_weighted', 'neg_log_loss', 'fowlkes_mallows_score', 'accuracy', 'neg_brier_score', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'f1_macro', 'jaccard_micro', 'explained_variance', 'jaccard_macro', 'neg_mean_gamma_deviance', 'recall_samples', 'roc_auc', 'neg_mean_absolute_error', 'v_measure_score', 'jaccard_weighted', 'neg_negative_likelihood_ratio', 'recall', 'neg_median_absolute_error', 'roc_auc_ovo_weighted', 'f1_samples', 'recall_micro', 'neg_root_mean_squared_log_error', 'neg_root_mean_squared_error', 'roc_auc_ovo', 'jaccard', 'positive_likelihood_ratio', 'recall_macro', 'jaccard_samples', 'f1_micro', 'neg_mean_absolute_percentage_error', 'recall_weighted', 'balanced_accuracy', 'top_k_accuracy', 'rand_score', 'f1', 'homogeneity_score', 'completeness_score',

In [33]:
gd.best_score_

np.float64(0.8384282217061075)

In [34]:
gd.best_params_

{'C': np.float64(705.0251256281407), 'penalty': 'l2'}

In [46]:
#2
max_depth=[3,4,5,6,7,8,9,10]
min_child_weight= [1,2,3,4,5,6,7]

hyperparams={
    "max_depth": max_depth,
    "min_child_weight": min_child_weight
}

gd = GridSearchCV(
    estimator=XGBClassifier(learning_rate=0.12, n_estimators=150, random_state=1, eval_metric="logloss"),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

Fitting 5 folds for each of 56 candidates, totalling 280 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=150,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
                         'min_child_weight': [1, 2, 3, 4, 5, 6, 7]},
             scoring='accuracy', verbose=True)

In [47]:
gd.best_score_

np.float64(0.8372669637813068)

In [48]:
gd.best_params_

{'max_depth': 6, 'min_child_weight': 1}

In [52]:
#3
import numpy as np

gamma=[i *0.1 for i in range(0,5)]
subsample= [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]
colsample_bytree= [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]
reg_alpha=[1e-5, 0.1, 1, 100]

hyperparams={
    "gamma":gamma,
    "sub_sample":subsample,
    "colsample_bytree": colsample_bytree,
    "reg_alpha": reg_alpha,    
}

gd=GridSearchCV(
    estimator=XGBClassifier(
        learning_rate=0.12, 
        n_estimators=150, 
        max_depth=6,
        min_child_weight=1,
        random_state=1, 
        eval_metric="logloss"
    ),
    param_grid=hyerparams,
    verbose=True,cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())

TypeError: Parameter grid for parameter 'C' needs to be a list or a numpy array, but got <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D3C8EDA7B0> (of type rv_continuous_frozen) instead. Single values need to be wrapped in a list with one element.

In [51]:
gd.best_score_

AttributeError: 'GridSearchCV' object has no attribute 'best_score_'

##교차검증 3가지

#Hold-out: train데이터셋에서 1개에서 특정 비율로 나눠서 한쪽을 가지고 학습하고 나머지를 가지고 적용해보며 검증
    -> overfitting(과적합)문제 발생 가능

#k-fold:주로 5개로 나눔
    -> train data를 5개로 나눠서 검증을 진행
    -> but, 하나의 교차검증 그룹내에서 원본 데이터의 내용이 제대로 안담길 수 있음 (ex: 원본에서는 남녀비율 6:4인데 샘플데이터에서 1:9가 될 수 있음)

stratified: 검증할때 원본데이터의 비율을 동일하게 가져감

In [ ]:
#하이퍼 파라미터 튜닝기법 3가지: Grid,Ramdom, BO
#SVC, LR,XGB, GBM

In [ ]:
#Random Forest

In [66]:
n_estimators = [10, 50, 100, 200]
max_depth= [3, None]
max_features = [0.1, 0.2, 0.5, 0.8, "sqrt", "log2"]
min_samples_split = [2,4,6,8,10] #부모 노드가 자식노드를 분리할 수 있는 자격요건(부모 노드 안에 포함된 데이터 숫자 기준) -> 현재 부모노드가 자식을 분리할 수 있을 만큼의 샘플갯수를 확보하고 있는가
min_samples_leaf = [2,4,6,8,10] #부모 노드의 답변(=자식노드)의 최소 샘플데이터 개수 기준

hyperparams = {
    "n_estimators":n_estimators,
    "max_depth" : max_depth,
    "max_features": max_features,
    "min_samples_split" : min_samples_split,
    "min_samples_leaf" : min_samples_leaf
}

gd= GridSearchCV(
    estimator=RandomForestClassifier(random_state=1),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance,train_answer.values.ravel())

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=1), n_jobs=-1,
             param_grid={'max_depth': [3, None],
                         'max_features': [0.1, 0.2, 0.5, 0.8, 'sqrt', 'log2'],
                         'min_samples_leaf': [2, 4, 6, 8, 10],
                         'min_samples_split': [2, 4, 6, 8, 10],
                         'n_estimators': [10, 50, 100, 200]},
             scoring='accuracy', verbose=True)

In [ ]:
gd.best_score_

In [ ]:
gd.best_params_

In [ ]:
#Extra Trees 모델

In [ ]:
n_estimators= [10,50,100,200]
max_depth= [3, None]
max_features = [0.1, 0.2, 0.5, 0.8, "sqrt", "log2"]
min_samples_split = [2,4,6,8,10] #부모 노드가 자식노드를 분리할 수 있는 자격요건(부모 노드 안에 포함된 데이터 숫자 기준) -> 현재 부모노드가 자식을 분리할 수 있을 만큼의 샘플갯수를 확보하고 있는가
min_samples_leaf = [2,4,6,8,10] #부모 노드의 답변(=자식노드)의 최소 샘플데이터 개수 기준

hyperparams = {
    "n_estimators":n_estimators,
    "max_depth" : max_depth,
    "max_features": max_features,
    "min_samples_split" : min_samples_split,
    "min_samples_leaf" : min_samples_leaf
}

gd= GridSearchCV(
    estimator=ExtraTreesClassifier(random_state=1),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance,train_answer.values.ravel())

In [ ]:
gd.best_score_

In [ ]:
gd.best_params_

In [ ]:
#현재 진행상황

#1. Kaffle 사이트 > raw data download (titanic)
#2. EDA > 탐색적 데이터 분석 과정 (생존, 사망 >  어떤 column 상관관계)
#3. Feature ENgineering (컴퓨터는 수치화된 데이터를 좋아하기 때문 )
#4. Model Process(KNN, Linear Regression, Logistic Regression, Baging, Boosting, Decision Tree, SVM,SVC, Random Forest 등등) 각 모델에 대한 설명
#5. Cross Validation (교차검증)
#6. High Chance & Important Features :교차검증 단계에서 모델이 학습한 결과값을 기반으로 테스트 데이터를 맞추기 전에 학습한 데이터를 가지고 다시 점검을 해봐서 score 도출
#   -> 이 score를 기반으로 도출 시, 어떤 feature에 높은 점수를 부여했는가가 나옴 > Decision Tree
#7. Hyper Parameter Tuning (각 모델들에게 전달할 매개변수 값, 검증된 방법 중 하나 , Grid Search, Random Search,)
#8. 각 모델들에게 필요한 하이퍼 파라미터를 어떻게 찾는지 학습
#9. 지금까지 학습했던 모델들에게 가장 최적의 파라미터값을 부여해서 학습 ->  각모델 학습 정확도 도출 -> 전체 평균을 내서 최종적인 결과값 도출 => kaggle 제출



In [ ]:
#voting => vote